# XCCNet baseline — Kermany pediatric chest X-rays

Reproduction of the classifier in Raghaw, Bhore, Rehman, and Kumar, *An Explainable Contrastive-based Dilated Convolutional Network with Transformer for Pediatric Pneumonia Detection*, arXiv:2410.16143 (2024).

**Runtime.** Upload this notebook to Google Colab and set `Runtime → Change runtime type → GPU`. The second code cell asks you to upload `baseline/xccnet.py` from the same folder on your computer.

**Data.** The full Guangzhou Women and Children’s Medical Center set from Kermany et al.: 1,583 normal and 4,273 pneumonia films. The notebook pools the Kaggle `train` / `val` / `test` folders and draws a stratified 80/10/10 split. That is the protocol in Section 4.1.1 of the paper. It is not the official 624-patient test.

**What is trained.** Dilated CNN (SFx, five blocks, dilation 2, filters 32–512) and a classification head on features concatenated with a CLIP ViT-B/32 encoder (CoTFx). The loss is Eq. 15: a contrastive term between the two branches plus binary cross-entropy. Preprocessing is bicubic resize to 256×256 and CLAHE (8×8 window, clip 2.0). A ChestX-Det PSPNet supplies the lung mask, because the Guangzhou images have no masks for the paper’s ResUNet++.

**What this run lines up with.** The paper’s “traditional augmentation” row (97.18% accuracy) plus lung masking. The published full stack (99.76%) also trains a style-vector GAN (2,624 synthetic normals) and a rib-suppression ResNet. The paper does not release those weights, the GAN layer sizes, or paired bone-suppressed targets, so those two stages are not invented here. Training images are class-balanced with a weighted sampler, which is the role the synthetic normals play in the paper, and geometric augmentation is applied only on the training split.

Adam is used, and the learning rate is multiplied by 0.3 when validation accuracy stalls, as in Section 4.1.4. The paper’s single learning rate is \(10^{-6}\). The spatial blocks are trained from scratch, so this notebook uses \(10^{-4}\) for those blocks and the classifier. The CLIP encoder stays at the OpenAI ViT-B/32 weights.

In [ ]:
import subprocess
import sys

packages = [
    "open_clip_torch",
    "torchxrayvision",
    "opencv-python-headless",
    "kagglehub",
    "matplotlib",
    "Pillow",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("installed", ", ".join(packages))

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/cv-project-sem7")
(DRIVE_ROOT / "baseline").mkdir(parents=True, exist_ok=True)
os.environ["KAGGLEHUB_CACHE"] = str(DRIVE_ROOT / "kagglehub")
print("saving under", DRIVE_ROOT)

CANDIDATES = [
    DRIVE_ROOT / "baseline",
    DRIVE_ROOT,
    Path.cwd(),
    Path.cwd() / "baseline",
    Path.cwd().parent,
    Path.cwd().parent / "baseline",
    Path("/content"),
    Path("/content/cv-project-sem7"),
    Path("/content/cv-project-sem7/baseline"),
]
found = None
for candidate in CANDIDATES:
    if (candidate / "xccnet.py").exists():
        found = candidate
        break
if found is None and "google.colab" in sys.modules:
    from google.colab import files
    print("Upload baseline/xccnet.py from your computer (the file next to this notebook).")
    uploaded = files.upload()
    if "xccnet.py" not in uploaded:
        raise FileNotFoundError("Upload the file named xccnet.py")
    Path("/content/xccnet.py").write_bytes(uploaded["xccnet.py"])
    shutil.copy("/content/xccnet.py", DRIVE_ROOT / "baseline" / "xccnet.py")
    found = DRIVE_ROOT / "baseline"
if found is None:
    raise FileNotFoundError(
        "Could not find xccnet.py. In Colab, upload baseline/xccnet.py when this cell asks for it."
    )
sys.path.insert(0, str(found))
import xccnet

ROOT = found.parent if found.name == "baseline" else found
if (found / "xccnet.py").exists() and found.name != "baseline" and (found / "baseline").exists():
    ROOT = found
print("xccnet:", found / "xccnet.py")
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("No GPU is visible. In Colab use Runtime → Change runtime type → GPU, then rerun from the top.")

In [ ]:
SEED = 42
BATCH_SIZE = 16
EPOCHS = 25
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
L1_LAMBDA = 1e-6
TEMPERATURE = 0.07
USE_LUNG_MASK = True

CACHE_DIR = DRIVE_ROOT / "baseline" / "cache" / "clahe_lung256"
CHECKPOINT = DRIVE_ROOT / "baseline" / "checkpoints" / "xccnet_best.pt"
RESULTS = DRIVE_ROOT / "baseline" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

xccnet.set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0

## Dataset

The images are the Kaggle mirror of Kermany et al. (`paultimothymooney/chest-xray-pneumonia`). If you already ran `kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")`, the next cell uses that cached folder. It asks for `kaggle.json` only when the cache is empty.

In [ ]:
def ensure_kaggle_credentials() -> None:
    kaggle_dir = Path.home() / ".kaggle"
    token = kaggle_dir / "kaggle.json"
    if token.exists() or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return
    if "google.colab" not in sys.modules:
        raise RuntimeError("Add a Kaggle API token in ~/.kaggle/kaggle.json, or set DATA_DIR.")
    from google.colab import files
    print("The dataset is not cached yet. Upload kaggle.json")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("kaggle.json was not uploaded")
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token.write_bytes(uploaded["kaggle.json"])
    token.chmod(0o600)

try:
    DATA_DIR = xccnet.resolve_kermany_root()
except Exception as error:
    text = str(error).lower()
    if not any(token in text for token in ("credential", "unauth", "401", "403", "kaggle.json")):
        raise
    ensure_kaggle_credentials()
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

records = xccnet.collect_kermany_records(DATA_DIR)
normal = sum(label == 0 for _, label in records)
pneumonia = sum(label == 1 for _, label in records)
print(f"root: {DATA_DIR}")
print(f"normal {normal}  pneumonia {pneumonia}  total {len(records)}")
print("paper counts: normal 1583  pneumonia 4273  total 5856")
if normal < 1500 or pneumonia < 4000:
    raise RuntimeError(
        "This folder is not the full Guangzhou set. Point DATA_DIR at the directory "
        "that contains the train, val, and test subfolders."
    )

In [ ]:
split = xccnet.stratified_split(records, seed=SEED)
counts = xccnet.split_counts(split)
for name, row in counts.items():
    print(f"{name:5s}  normal {row['normal']:4d}  pneumonia {row['pneumonia']:4d}  total {row['total']:4d}")

segmenter = None
if USE_LUNG_MASK:
    try:
        segmenter = xccnet.LungSegmenter(DEVICE)
        print("lung mask: ChestX-Det PSPNet (left lung ∪ right lung)")
    except Exception as error:
        print(f"lung mask unavailable ({error}). Continuing with CLAHE only.")
        segmenter = None

def _progress(done: int, total: int) -> None:
    if done == total or done % 200 == 0:
        print(f"preprocess {done}/{total}", flush=True)

prepared = {}
for name, rows in split.items():
    prepared[name] = xccnet.preprocess_records(
        rows,
        CACHE_DIR,
        segmenter=segmenter,
        batch_size=8,
        progress=_progress,
    )
print("cache:", CACHE_DIR)

In [ ]:
from PIL import Image
import numpy as np

preview = prepared["train"][:4]
figure, axes = plt.subplots(1, 4, figsize=(12, 3))
for axis, (path, label) in zip(axes, preview):
    axis.imshow(np.asarray(Image.open(path)), cmap="gray")
    axis.set_title("normal" if label == 0 else "pneumonia")
    axis.axis("off")
figure.suptitle("Training images after resize, CLAHE, and the lung mask")
figure.tight_layout()
figure.savefig(RESULTS / "preprocess_preview.png", dpi=120)
plt.show()

## Train

This is the run that reached 94.87% test accuracy. CLAHE and the ChestX-Det lung mask are on. CLIP ViT-B/32 stays frozen. The dilated CNN and the classifier train at \(10^{-4}\) with batch size 16. Checkpoints and figures are written to Google Drive.

In [ ]:
train_loader = xccnet.make_loader(
    prepared["train"], train=True, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
val_loader = xccnet.make_loader(
    prepared["val"], train=False, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
test_loader = xccnet.make_loader(
    prepared["test"], train=False, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

model = xccnet.build_clip(DEVICE, freeze=True)
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
frozen = sum(parameter.numel() for parameter in model.parameters() if not parameter.requires_grad)
print(f"trainable parameters {trainable:,}  frozen CLIP parameters {frozen:,}")

fit = xccnet.fit(
    model,
    train_loader,
    val_loader,
    DEVICE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    temperature=TEMPERATURE,
    l1_lambda=L1_LAMBDA,
    patience=PATIENCE,
    checkpoint_path=CHECKPOINT,
)
print("best checkpoint:", fit.best_path)
print("best validation:", {key: round(fit.best_val[key], 2) for key in ("accuracy", "precision", "recall", "f1")})

In [ ]:
test_metrics = xccnet.evaluate(model, test_loader, DEVICE)
keys = ("accuracy", "precision", "recall", "f1")
print("test", {key: round(test_metrics[key], 2) for key in keys})
print(
    f"confusion  tp {test_metrics['tp']}  fp {test_metrics['fp']}  "
    f"tn {test_metrics['tn']}  fn {test_metrics['fn']}  n {test_metrics['n']}"
)
print()
print(f"{'setting':<32} {'acc':>8} {'prec':>8} {'rec':>8} {'f1':>8}")
for name, row in xccnet.PAPER_METRICS.items():
    print(
        f"{name:<32} {row['accuracy']:8.2f} {row['precision']:8.2f} "
        f"{row['recall']:8.2f} {row['f1']:8.2f}"
    )
print(
    f"{'this run (test)':<32} {test_metrics['accuracy']:8.2f} {test_metrics['precision']:8.2f} "
    f"{test_metrics['recall']:8.2f} {test_metrics['f1']:8.2f}"
)

history_path = RESULTS / "metrics.json"
payload = {
    "paper": "arXiv:2410.16143",
    "dataset": "Kermany Guangzhou via Kaggle paultimothymooney/chest-xray-pneumonia",
    "split": "stratified 80/10/10 on the pooled real images",
    "seed": SEED,
    "counts": counts,
    "lung_mask": segmenter is not None,
    "learning_rate": LEARNING_RATE,
    "clip_frozen": True,
    "temperature": TEMPERATURE,
    "paper_table": xccnet.PAPER_METRICS,
    "best_validation": {key: fit.best_val[key] for key in keys},
    "test": {key: test_metrics[key] for key in (*keys, "tp", "tn", "fp", "fn", "n")},
    "history": fit.history,
}
history_path.write_text(json.dumps(payload, indent=2))
print("wrote", history_path)

figure, axis = plt.subplots(figsize=(6, 3.5))
axis.plot([row["epoch"] for row in fit.history], [row["val_accuracy"] for row in fit.history], marker="o")
axis.axhline(97.18, color="0.4", linestyle="--", label="paper test, traditional aug 97.18")
axis.axhline(99.76, color="0.15", linestyle=":", label="paper test, full stack 99.76")
axis.set_xlabel("epoch")
axis.set_ylabel("validation accuracy (%)")
axis.legend(fontsize=8)
figure.tight_layout()
figure.savefig(RESULTS / "val_accuracy.png", dpi=120)
plt.show()

## Grad-CAM

The map is taken from the last dilated convolution in SFx, which is the spatial branch the paper visualizes with Grad-CAM. Redder regions contributed more to the pneumonia logit.

In [ ]:
samples = []
for path, label in prepared["test"]:
    if len(samples) >= 4:
        break
    if label == (len(samples) % 2):
        samples.append((path, label))
if len(samples) < 4:
    samples = prepared["test"][:4]

figure, axes = plt.subplots(2, 4, figsize=(12, 6))
for column, (path, label) in enumerate(samples):
    image = np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0
    tensor = torch.from_numpy(image).unsqueeze(0).repeat(3, 1, 1)
    camera, probability = xccnet.grad_cam(model, tensor, positive=True)
    camera = np.array(Image.fromarray((camera * 255).astype(np.uint8)).resize((image.shape[1], image.shape[0])))
    axes[0, column].imshow(image, cmap="gray")
    axes[0, column].set_title("normal" if label == 0 else "pneumonia")
    axes[0, column].axis("off")
    axes[1, column].imshow(image, cmap="gray")
    axes[1, column].imshow(camera, cmap="jet", alpha=0.45)
    axes[1, column].set_title(f"p(pneumonia)={probability:.2f}")
    axes[1, column].axis("off")
figure.tight_layout()
figure.savefig(RESULTS / "gradcam.png", dpi=120)
plt.show()